# PP-OCRv6_small_rec 파인튜닝 (Colab GPU)

실행 전 런타임을 GPU로 바꿔라: 상단 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`.

순서대로 셀을 실행하면 됨. 3번 셀에서 로컬에서 만든 `train_data_colab.zip`을 업로드해야 함.

In [ ]:
# 1. 의존성 설치 (ppocr/data/imaug 전체 import 체인 기준 필요한 것 다 포함)
# paddlepaddle-gpu(1.4GB, bcebos 느림)는 드라이브에 캐싱 — 최초 1회만 다운로드,
# 그 다음 세션부턴 드라이브에서 바로 설치(네트워크 안 타서 느림/타임아웃 문제 자체가 없어짐)
from google.colab import drive
drive.mount('/content/drive')

import glob, os
WHEEL_DIR = '/content/drive/MyDrive/paddle_wheels'
os.makedirs(WHEEL_DIR, exist_ok=True)
existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
if existing:
    print(f'캐시 사용: {existing[0]}')
    !pip install "{existing[0]}" -q
else:
    print('캐시 없음 - 최초 1회 다운로드 (다음부턴 훨씬 빨라짐)')
    !pip download --timeout 300 --retries 5 paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ -d {WHEEL_DIR} --no-deps
    existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
    !pip install "{existing[0]}" -q

!pip install lmdb scikit-image albumentations opencv-python pyclipper shapely rapidfuzz -q

In [ ]:
# 2. PaddleOCR 클론(--depth 1: 전체 히스토리 없이 최신 코드만, 훨씬 빠름) + 사전학습 가중치 다운로드
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!mkdir -p pretrain_weights
!wget -q https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams -O pretrain_weights/PP-OCRv6_small_rec_pretrained.pdparams
print('done')

In [ ]:
# 3. 학습데이터 업로드 (로컬의 data/generated/train_data_colab.zip 선택)
from google.colab import files
uploaded = files.upload()
!mkdir -p train_data
!unzip -q train_data_colab.zip -d train_data
!wc -l train_data/train_list.txt train_data/val_list.txt

In [ ]:
# 4. 파인튜닝 config 작성 (GPU용 — 배치사이즈/워커 원복, use_gpu:true)
config_yaml = '''
Global:
  model_name: PP-OCRv6_small_rec
  debug: false
  use_gpu: true
  epoch_num: 20
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/PP-OCRv6_small_rec_finetune
  save_epoch_step: 5
  eval_batch_step: [0, 500]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_weights/PP-OCRv6_small_rec_pretrained
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img: doc/imgs_words/ch/word_1.jpg
  character_dict_path: ppocr/utils/dict/ppocrv6_dict.txt
  max_text_length: &max_text_length 25
  infer_mode: false
  use_space_char: true
  distributed: true
  save_res_path: ./output/rec/predicts_ppocrv6_small.txt
  d2s_train_image_shape: [3, 48, 320]


Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05


Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV4
    model_size: small
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: lightsvtr
            dims: 120
            depth: 2
            mlp_ratio: 2.0
            local_kernel: 7
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: MultiScaleDataSet
    ds_width: false
    data_dir: ./train_data/
    ext_op_transform_idx: 1
    label_file_list:
    - ./train_data/train_list.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - RecConAug:
        prob: 0.5
        ext_data_num: 2
        image_shape: [48, 320, 3]
        max_text_length: *max_text_length
    - RecAug:
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  sampler:
    name: MultiScaleSampler
    scales: [[320, 32], [320, 48], [320, 64]]
    first_bs: &bs 128
    fix_bs: false
    divided_factor: [8, 16]
    is_training: True
  loader:
    shuffle: true
    batch_size_per_card: *bs
    drop_last: true
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./train_data
    label_file_list:
    - ./train_data/val_list.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 128
    num_workers: 4
'''
with open('configs/rec/PP-OCRv6/PP-OCRv6_small_rec_finetune_colab.yml', 'w') as f:
    f.write(config_yaml)
print('config written')

In [ ]:
# 5. 학습 실행
!python tools/train.py -c configs/rec/PP-OCRv6/PP-OCRv6_small_rec_finetune_colab.yml

In [ ]:
# 6. (학습 끝난 후) 결과물을 구글드라이브나 로컬로 백업
# Colab은 세션 끊기면 output/ 날아감 - 꼭 백업할 것
from google.colab import drive
drive.mount('/content/drive')
!cp -r output /content/drive/MyDrive/PP-OCRv6_small_rec_finetune_output
print('backed up to Google Drive')